# Generación de Base de Datos Vectorial para el agente

## Importacion de recetas

In [1]:
import pandas as pd

In [ ]:
recipes_data = pd.read_json('./recipes.json')

In [3]:
recipes_data.head()

,name,image_url,recipe_id,calories,protein,carbs,fat,meal_types,diet_types,ingredients,recipe_url
0,lowfat berry blue frozen dessert,https://img.sndimg.com/food/image/upload/w_555...,0,42.72,0.80,9.28,0.62,[snack],"[high_fiber, low_calorie, low_carb, vegetarian]","blueberries, granulated sugar, vanilla yogurt,...",https://www.food.com/recipe/recipe-0
1,biryani,https://img.sndimg.com/food/image/upload/w_555...,1,185.12,10.57,14.07,9.80,"[dinner, lunch]","[high_fiber, low_calorie, low_carb]","saffron, milk, hot green chili peppers, onions...",https://www.food.com/recipe/recipe-1
2,best lemonade,https://img.sndimg.com/food/image/upload/w_555...,2,77.78,0.08,20.38,0.05,[snack],"[high_fiber, low_calorie, vegan]","sugar, lemons, rind of, lemon, zest of, fresh ...",https://www.food.com/recipe/recipe-2
3,carinas tofuvegetable kebabs,https://img.sndimg.com/food/image/upload/w_555...,3,268.05,14.65,32.10,12.00,[lunch],"[high_fiber, low_calorie, vegan]","extra firm tofu, eggplant, zucchini, mushrooms...",https://www.food.com/recipe/recipe-3
4,cabbage soup,https://img.sndimg.com/food/image/upload/w_555...,4,25.90,1.08,6.28,0.10,[snack],"[high_fiber, low_calorie, low_carb, vegan]","plain tomato juice, cabbage, onion, carrots, c...",https://www.food.com/recipe/recipe-4


In [4]:
recipes_list = recipes_data.to_dict(orient='records')

In [5]:
from langchain_core.documents import Document
from tqdm import tqdm

In [6]:
def prepare_documents(data):
    docs = []
    for item in data:
        diet = ", ".join(item['diet_types']) if isinstance(item['diet_types'], list) else item['diet_types']

        # El contenido es lo que el LLM "leerá" para comparar
        page_content = f"Name: {item['name']}. Diet Types: {diet}. Ingredients: {item['ingredients']}."

        # El metadata sirve para hacer cálculos matemáticos de macros
        metadata = {
            "calories": item["calories"],
            "protein": item["protein"],
            "carbs": item["carbs"],
            "fat": item["fat"],
            "meal_types": item["meal_types"][0] if isinstance(item['meal_types'], list) and item["meal_types"] else "any"
        }
        docs.append(Document(page_content=page_content, metadata=metadata))
    return docs

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
import os

In [ ]:
PERSIST_DIR = "./chroma_db_recipes"
embeddings = OllamaEmbeddings(
    model="nomic-embed-text", # Es rápido y eficiente 
    base_url="http://localhost:11434" # Asegúrate de que este puerto esté mapeado
)

if os.path.exists(PERSIST_DIR):
    print("Cargando base de datos existente...")
    # Si ya existe, la cargamos desde el disco
    vector_db = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
else:
    print("Creando nueva base de datos...")
    # Si no existe, creamos una nueva vacía
    vector_db = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)

Creando nueva base de datos...


C:\Users\Zeta\AppData\Local\Temp\ipykernel_8612\610580744.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)


In [ ]:
# PROCESAMIENTO POR LOTES Y GUARDADO
batch_size = 512 # Ajusta este valor según tu GPU y memoria disponible
total_records = len(recipes_list)

for i in tqdm(range(0, total_records, batch_size), desc="Progreso Total"):
    batch = recipes_list[i:i + batch_size]
    
    # 1. Preparar rápido (formateo)
    docs = prepare_documents(batch)
    
    # 2. Insertar en la BD (esto es lo que tarda la GPU)
    vector_db.add_documents(docs)

print(f"\nBase de datos guardada en: {PERSIST_DIR}")

Progreso Total:   2%|▏         | 6/356 [00:15<14:10,  2.43s/it]